In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import bidsio
import sys
import copy
import pickle
sys.path.append('../')
from helpers import *
from fns_ffn import *


device = 'cuda:5' if torch.cuda.is_available() else 'cpu'
seed, RES = 0, 64
bids_loader = bidsio.BIDSLoader(data_entities=[{'subject': '',
                                              'session': '',
                                              'suffix': 'T1w',
                                              'space': 'MNI152NLin2009aSym'}],
                              target_entities=[],
                              data_derivatives_names=['ATLAS'],
                              batch_size=1,
                              root_dir='./atlas/data/test/')


In [ ]:
idx = 131
mask = np.fft.fftshift(np.ones((RES, RES, RES))).astype(np.complex64)
tmp = bids_loader.load_sample(idx = idx, data_only=True) / 255.0
signal = resize(tmp, (1, RES, RES, RES))[0] 
plt.imshow(signal[:,:,24])
plt.colorbar()
plt.show()

In [ ]:
learning_rate, iters = 2e-3, 2000
rng = np.random.default_rng(seed)
slice_idx = 24

# varying mapping_size
# model_params = {
#     'mapping_size': -1,
#     'width': 400,
#     'num_layers': 1,
#     'scale': 5.,
# }

# varying width
model_params = {
    'mapping_size': 1520,
    'width': -1,
    'num_layers': 1,
    'scale': 5.,
}

model_sizes = [52000, 105000]
outputs = {}
to_save_outputs = {}
if model_params['mapping_size'] == -1:
    param_to_vary = 'mapping_size'
else:    
    param_to_vary = 'width'
for model_size in model_sizes:
    model_params[param_to_vary] = compute_params_from_model_size("ffn_eta", model_params, model_size)
    print(f'{param_to_vary}: {model_params[param_to_vary]}')
    mapping_size, scale = model_params['mapping_size'], model_params['scale']
    num_layers, width = model_params['num_layers'], model_params['width']
    torch.manual_seed(seed)
    B = torch.randn(mapping_size, len(signal.shape), dtype=torch.float32, device=device) * scale
    output = fit_fourier_features_learn(y_gt=signal, B=B, network_size=(num_layers, width), iters=iters,
                                                               learning_rate=learning_rate,
                                                               log_interval=1000, seed=seed, device=device, mask=mask, count_params=True)
    outputs[f'{model_params[param_to_vary]}'] = output     
    to_save_outputs[f'{model_params[param_to_vary]}'] = output['best_pred'].reshape((RES, RES, RES))[:,:,slice_idx]
    error = np.linalg.norm(signal.flatten() - output['best_pred'].flatten())                       
    print(f"Error: {error:.3e}, Loss: {output['best_loss']:.3e}")
    model_params[param_to_vary] = -1

with open(f"3d_mri/ffn.pkl", "wb") as f:
    pickle.dump(to_save_outputs, f)

In [ ]:
from helpers import *

slice_idx = 24 
def slice_outputs(outputs, slice_idx):
    new_outputs = copy.deepcopy(outputs)
    small_param, large_param = list(outputs.keys())
    small_pred, large_pred   = new_outputs[small_param]['best_pred'].reshape((RES, RES, RES)), new_outputs[large_param]['best_pred'].reshape((RES, RES, RES))
    new_outputs[small_param], new_outputs[large_param] = new_outputs[small_param], new_outputs[large_param]
    new_outputs[small_param]['best_pred'] = small_pred[:,:,slice_idx]
    new_outputs[large_param]['best_pred'] = large_pred[:,:,slice_idx]
    return new_outputs

plot_error_heatmaps(signal[:,:,slice_idx], slice_outputs(outputs, slice_idx), model_name="FFN")